# Preprocessing

In [5]:
import pandas as pd

test_pairs = pd.read_csv("/content/drive/MyDrive/Assignment 2 - Rating Prediction/test_pairs.csv")
train = pd.read_csv("/content/drive/MyDrive/Assignment 2 - Rating Prediction/train.csv")
users = pd.read_csv("/content/drive/MyDrive/Assignment 2 - Rating Prediction/users.csv")
movies = pd.read_csv("/content/drive/MyDrive/Assignment 2 - Rating Prediction/movies.csv")

print("Shape of training data:", train.shape)
print("Shape of test pair data", test_pairs.shape)
print("Shape of user metadata", users.shape)
print("Movies:", movies.shape)



Shape of training data: (797758, 4)
Shape of test pair data (202451, 2)
Shape of user metadata (6040, 4)
Movies: (3883, 4)


In [6]:
# Check for duplicates and null values
print("Duplicates:", train.duplicated().sum())
print("NULL:", train.isnull().sum())
print("Duplicates:", users.duplicated().sum())
print("NULL:", users.isnull().sum())
print("Duplicates:", movies.duplicated().sum())
print("NULL:", movies.isnull().sum())


Duplicates: 0
NULL: user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Duplicates: 0
NULL: user_id       0
gender        0
age_group     0
occupation    0
dtype: int64
Duplicates: 0
NULL: item_id    0
title      0
year       0
genres     0
dtype: int64


In [7]:
# Look at the distrbution of all ratings in training dataset
train.describe()
print(train['rating'].value_counts())

rating
4    280133
3    204611
5    189059
2     81748
1     42207
Name: count, dtype: int64


We need to create an 80:20 temporal split, as well as calculate how many ratings 20% is for every single user and then sort their ratings by timestamps.

In [8]:
import numpy as np

#Create the 80:20 temporal split as described in the discussion
def temporal_split(df, test_ratio=0.20):
    # Sort each user's rating by time
    df = df.sort_values(['user_id', 'timestamp'], ascending=[True, True])

    # Create the training set and validation set
    train_split = []
    val_split = []
    for user_id, group in df.groupby('user_id'):
        ratings_count = len(group)

        # Calculate how many rating we should keep
        test_count = int(np.ceil(test_ratio * ratings_count))
        train_count = ratings_count - test_count

        # Map the positions back to the original dataframe indices
        user_set = group.index.tolist()

        # Create the training and validation test
        train_split.extend(user_set[:train_count])
        val_split.extend(user_set[train_count:])

    # Create data frame
    train_df = df.loc[train_split].copy()
    val_df = df.loc[val_split].copy()

    return train_df, val_df

In [11]:
import sklearn
train_df, val_df = temporal_split(train)

# Calculate the global mean from your training split
global_mean = train_df['rating'].mean()
val_df['pred_baseline'] = global_mean
print("Global mean:", global_mean)
print(f"Baseline RMSE: {rmse(val_df['rating'], val_df['pred_baseline'])}")

Global mean: 3.643977015146534


NameError: name 'rmse' is not defined

This is where we begin the hybridization process.

In [ ]:
def preprocess_movies(movies_df):
    # Split the pipe-separated genres [cite: 50]
    genres = movies_df['genres'].str.get_dummies(sep='|')
    return pd.concat([movies_df[['item_id', 'year']], genres], axis=1)

# Demographic encoding for Users [cite: 54]
def preprocess_users(users_df):
    users_df['is_male'] = (users_df['gender'] == 'M').astype(int)
    # One-hot encode occupation since it has 21 labels [cite: 54]
    occupations = pd.get_dummies(users_df['occupation'], prefix='occ')
    return pd.concat([users_df[['user_id', 'age_group']], occupations, users_df[['is_male']]], axis=1)

In [ ]:
from scipy.sparse import csr_matrix

def create_interaction_matrix(train_df):
    # Map IDs to contiguous indices for matrix operations [cite: 44, 47]
    user_map = {id: i for i, id in enumerate(train_df['user_id'].unique())}
    item_map = {id: i for i, id in enumerate(train_df['item_id'].unique())}

    row = train_df['user_id'].map(user_map).values
    col = train_df['item_id'].map(item_map).values
    data = train_df['rating'].values

    # CSR matrix format: (data, (row_ind, col_ind))
    matrix = csr_matrix((data, (row, col)))
    return matrix, user_map, item_map

In [ ]:
def get_cold_start_items(train_df, test_pairs_df):
    train_items = set(train_df['item_id'].unique())
    test_items = set(test_pairs_df['item_id'].unique())
    cold_items = test_items - train_items # These 40 items need fallback [cite: 153]
    return cold_items